In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-preparation").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 14:35:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill: Point-of-Sale Sales Aggregation

Consume a stream of point-of-sale records with format `product_name, sale_timestamp`.  
Aggregate **how many sales each product had in each 10-minute window**, using a 1-minute watermark.

**Your task:**
1. Generate or load sample POS data as a file stream.  
   Each record has the format: `product, sale_timestamp` (e.g., `Laptop,2024-01-01 09:01:30`).
2. Parse the CSV stream with the correct schema.
3. Apply a `withWatermark("saleTime", "1 minute")` and group by product and a 10-minute tumbling window.
4. Count sales per product per window.
5. Write results to CSV and display them.

---

In [2]:
# ── Data generator: create POS sales events ──────────────────────────────
import datetime, random

random.seed(7)
PRODUCTS = ["Laptop", "Phone", "Tablet", "Headphones", "Charger"]
POS_DIR  = "tmp/pos_stream"

os.makedirs(POS_DIR, exist_ok=True)

BASE_TS = datetime.datetime(2024, 1, 1, 9, 0, 0)

def gen_sales(start_offset_s, count, filename):
    lines = []
    for i in range(count):
        product  = random.choice(PRODUCTS)
        sale_ts  = BASE_TS + datetime.timedelta(seconds=start_offset_s + i * 30)
        lines.append(f"{product},{sale_ts.strftime("%Y-%m-%d %H:%M:%S")}")
    with open(f"{POS_DIR}/{filename}", "w") as f:
        f.write("\n".join(lines) + "\n")
    return lines

# Create 3 batches simulating arrivals over ~25 minutes
gen_sales(0,   30, "pos_batch_01.csv")   # 09:00 - 09:14:30
gen_sales(900, 20, "pos_batch_02.csv")   # 09:15 - 09:24:30
gen_sales(1500, 15, "pos_batch_03.csv")  # 09:25 - 09:32:00

total = sum(len(f.readlines()) for f in
            [open(f"{POS_DIR}/pos_batch_{n:02d}.csv") for n in range(1, 4)])
print(f"Generated {total} sales records across 3 batches.")

# Preview
with open(f"{POS_DIR}/pos_batch_01.csv") as f:
    for line in list(f)[:5]:
        print(line, end="")

Generated 65 sales records across 3 batches.
Tablet,2024-01-01 09:00:00
Phone,2024-01-01 09:00:30
Headphones,2024-01-01 09:01:00
Laptop,2024-01-01 09:01:30
Laptop,2024-01-01 09:02:00


In [ ]:
# ── Your solution here ────────────────────────────────────────────────────
# 1. Define a schema with "product" (StringType) and "saleTime" (TimestampType)
# 2. readStream from POS_DIR
# 3. withWatermark + groupBy(product, window("saleTime", "10 minutes")) + count()
# 4. writeStream to "tmp/pos_out", trigger(availableNow=True)
# 5. Read and show the result




# Shutdown Spark when done

In [ ]:
spark.stop()